In [1]:
# ── Imports ──────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported!")

Matplotlib is building the font cache; this may take a moment.


✅ All libraries imported!


In [7]:
# Load YOUR dataset
df = pd.read_csv('../data/SampleSuperstore.csv', encoding='latin1')

print("✅ Data Loaded!")
print(f"📊 Shape: {df.shape}")           # (9994, 13)
print(f"📋 Columns: {df.columns.tolist()}")

✅ Data Loaded!
📊 Shape: (9994, 13)
📋 Columns: ['Ship Mode', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Category', 'Sub-Category', 'Sales', 'Quantity', 'Discount', 'Profit']


In [8]:
# First look at data
print("=== FIRST 5 ROWS ===")
display(df.head())

print("\n=== DATA TYPES ===")
print(df.dtypes)

print("\n=== MISSING VALUES ===")
print(df.isnull().sum())
# Great news: YOUR data has ZERO missing values!

=== FIRST 5 ROWS ===


,Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit
0,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,261.9600,2,0.00,41.9136
1,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,731.9400,3,0.00,219.5820
2,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,14.6200,2,0.00,6.8714
3,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,957.5775,5,0.45,-383.0310
4,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,22.3680,2,0.20,2.5164



=== DATA TYPES ===
Ship Mode        object
Segment          object
Country          object
City             object
State            object
Postal Code       int64
Region           object
Category         object
Sub-Category     object
Sales           float64
Quantity          int64
Discount        float64
Profit          float64
dtype: object

=== MISSING VALUES ===
Ship Mode       0
Segment         0
Country         0
City            0
State           0
Postal Code     0
Region          0
Category        0
Sub-Category    0
Sales           0
Quantity        0
Discount        0
Profit          0
dtype: int64


In [9]:
# ── Feature Engineering ──────────────────────────────
# Profit Margin = how much profit per dollar of sales
df['Profit_Margin'] = round((df['Profit'] / df['Sales']) * 100, 2)

# Revenue per Unit
df['Revenue_Per_Unit'] = round(df['Sales'] / df['Quantity'], 2)

# Is it a loss? (True/False flag)
df['Is_Loss'] = df['Profit'] < 0

# Discount Bracket (group discounts into buckets)
df['Discount_Bracket'] = pd.cut(
    df['Discount'],
    bins=[-0.01, 0, 0.2, 0.4, 0.6, 0.8],
    labels=['No Discount', '1-20%', '21-40%', '41-60%', '61-80%']
)

# Save cleaned data
df.to_csv('../data/cleaned_superstore.csv', index=False)

print("✅ Features engineered!")
print(df[['Sales','Profit','Profit_Margin','Is_Loss','Discount_Bracket']].head())

✅ Features engineered!
      Sales    Profit  Profit_Margin  Is_Loss Discount_Bracket
0  261.9600   41.9136          16.00    False      No Discount
1  731.9400  219.5820          30.00    False      No Discount
2   14.6200    6.8714          47.00    False      No Discount
3  957.5775 -383.0310         -40.00     True           41-60%
4   22.3680    2.5164          11.25    False            1-20%


In [10]:
# ── KPI Summary ──────────────────────────────────────
total_sales    = df['Sales'].sum()
total_profit   = df['Profit'].sum()
total_orders   = len(df)
avg_margin     = df['Profit_Margin'].mean()
loss_orders    = df['Is_Loss'].sum()
loss_pct       = round((loss_orders / total_orders) * 100, 1)

print("=" * 45)
print("       📊 BUSINESS HEALTH REPORT")
print("=" * 45)
print(f"  💰 Total Sales       : ${total_sales:,.2f}")
print(f"  📈 Total Profit      : ${total_profit:,.2f}")
print(f"  📦 Total Orders      : {total_orders:,}")
print(f"  📉 Avg Profit Margin : {avg_margin:.2f}%")
print(f"  🔴 Loss Orders       : {loss_orders} ({loss_pct}%)")
print("=" * 45)

       📊 BUSINESS HEALTH REPORT
  💰 Total Sales       : $2,297,199.86
  📈 Total Profit      : $286,394.05
  📦 Total Orders      : 9,994
  📉 Avg Profit Margin : 12.03%
  🔴 Loss Orders       : 1871 (18.7%)


In [11]:
# ── Category Analysis ────────────────────────────────
category_summary = df.groupby('Category').agg(
    Total_Sales    = ('Sales', 'sum'),
    Total_Profit   = ('Profit', 'sum'),
    Total_Orders   = ('Sales', 'count'),
    Avg_Margin     = ('Profit_Margin', 'mean'),
    Loss_Count     = ('Is_Loss', 'sum')
).reset_index()

category_summary['Profit_per_Order'] = round(
    category_summary['Total_Profit'] / category_summary['Total_Orders'], 2
)

print("📊 Category Summary:")
display(category_summary)

📊 Category Summary:


,Category,Total_Sales,Total_Profit,Total_Orders,Avg_Margin,Loss_Count,Profit_per_Order
0,Furniture,741998.7953,18451.2728,2121,3.878472,714,8.70
1,Office Supplies,719047.0320,122490.8008,6026,13.803032,886,20.33
2,Technology,836154.0330,145451.9773,1847,15.611256,271,78.75


In [12]:
# ── Sub-Category Deep Dive ───────────────────────────
sub_cat = df.groupby(['Category', 'Sub-Category']).agg(
    Total_Sales   = ('Sales', 'sum'),
    Total_Profit  = ('Profit', 'sum'),
    Total_Qty     = ('Quantity', 'sum'),
    Avg_Discount  = ('Discount', 'mean'),
    Avg_Margin    = ('Profit_Margin', 'mean'),
    Orders        = ('Sales', 'count'),
    Loss_Orders   = ('Is_Loss', 'sum')
).reset_index().sort_values('Total_Profit')

# Loss percentage per sub-category
sub_cat['Loss_Rate'] = round(
    (sub_cat['Loss_Orders'] / sub_cat['Orders']) * 100, 1
)

print("🔴 CONFIRMED DEAD ZONES (Negative Profit):")
display(sub_cat[sub_cat['Total_Profit'] < 0])

print("\n🟢 TOP PERFORMERS:")
display(sub_cat.nlargest(5, 'Total_Profit'))

🔴 CONFIRMED DEAD ZONES (Negative Profit):


,Category,Sub-Category,Total_Sales,Total_Profit,Total_Qty,Avg_Discount,Avg_Margin,Orders,Loss_Orders,Loss_Rate
3,Furniture,Tables,206964.5320,-17725.4811,1241,0.261285,-14.771755,319,203,63.6
0,Furniture,Bookcases,114879.9963,-3472.5560,868,0.211140,-12.663904,228,109,47.8
12,Office Supplies,Supplies,46673.5380,-1189.0995,647,0.076842,11.203947,190,33,17.4



🟢 TOP PERFORMERS:


,Category,Sub-Category,Total_Sales,Total_Profit,Total_Qty,Avg_Discount,Avg_Margin,Orders,Loss_Orders,Loss_Rate
14,Technology,Copiers,149528.030,55617.8249,234,0.161765,31.719265,68,0,0.0
16,Technology,Phones,330007.054,44513.7306,3289,0.154556,11.919899,889,136,15.3
13,Technology,Accessories,167380.318,41935.6649,2976,0.078452,21.817510,775,91,11.7
10,Office Supplies,Paper,78479.206,34053.5693,5178,0.074891,42.560036,1370,0,0.0
6,Office Supplies,Binders,203412.733,30221.7633,5974,0.372292,-19.959508,1523,613,40.2
